In [1]:
!git clone https://github.com/vinhVVN/Realized-Volatility-Prediction.git

Cloning into 'Realized-Volatility-Prediction'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 96 (delta 34), reused 84 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 59.58 KiB | 5.42 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [2]:
import os
import sys
import gc
import yaml
import numpy as np
import pandas as pd
import warnings
import lightgbm as lgb

warnings.filterwarnings('ignore')

# 1. Định vị chính xác thư mục root dự án sau khi Git Clone trên Kaggle
PROJECT_ROOT = '/kaggle/working/Realized-Volatility-Prediction'

if os.path.exists(PROJECT_ROOT):
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"🚀 Thư mục làm việc hiện tại của hệ thống: {os.getcwd()}")

# 2. Cài đặt bổ sung thư viện nếu cần (Kaggle đã có sẵn đa số)
!pip install fastparquet -q

🚀 Thư mục làm việc hiện tại của hệ thống: /kaggle/working/Realized-Volatility-Prediction
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.6 MB/s eta 0:00:00


In [3]:
from src.training.cv_splitter import TimeSeriesCVSplitter
from src.training.metrics import rmspe, feval_rmspe

In [4]:
# Đường dẫn đọc file đặc trưng tổng hợp đã xuất ra từ mốc trước
FEATURES_PATH = '/kaggle/input/notebooks/thnhvinhs/notebook22d94af6ba/Realized-Volatility-Prediction/data/processed/features_FINAL.feather'

print("📊 Đang nạp ma trận đặc trưng khổng lồ...")
df = pd.read_feather(FEATURES_PATH)
print(f"Kích thước tập dữ liệu huấn luyện: {df.shape}")

# Khởi tạo bộ chia fold chuỗi thời gian
splitter = TimeSeriesCVSplitter(n_splits=4, random_state=42)

# Khôi phục trình tự thời gian toàn cục nếu file chưa được map
if 'true_time_id' not in df.columns:
    print("⏳ Đang chạy t-SNE 2 vòng để khôi phục trật tự dòng thời gian toàn cục...")
    time_order = splitter.reverse_engineer_time_order(df, df)
    df = df.merge(time_order, on='time_id', how='left')

# Loại bỏ các cột định danh hoặc nhãn mục tiêu ra khỏi danh sách đặc trưng học tập
not_features = ['row_id', 'time_id', 'stock_id', 'target', 'true_time_id', 'tsne_val']
features = [col for col in df.columns if col not in not_features]

print(f"✅ Tổng số lượng đặc trưng tinh túy đưa vào LightGBM: {len(features)} cột.")

📊 Đang nạp ma trận đặc trưng khổng lồ...
Kích thước tập dữ liệu huấn luyện: (428932, 574)
⏳ Đang chạy t-SNE 2 vòng để khôi phục trật tự dòng thời gian toàn cục...
2026-06-08 13:25:31 | INFO     | cv_splitter | Đang khôi phục trật tự thời gian bằng t-SNE...
2026-06-08 13:31:27 | INFO     | cv_splitter | Hoàn tất khôi phục trật tự thời gian.
✅ Tổng số lượng đặc trưng tinh túy đưa vào LightGBM: 571 cột.


In [5]:
# Thiết lập các siêu tham số cốt lõi cho mô hình Cây
params = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,      # Tỷ lệ học nhỏ để mô hình hội tụ mượt mà, hạn chế học vẹt
    'num_leaves': 31,
    'max_depth': -1,
    'min_data_in_leaf': 500,     # Rất quan trọng để tránh các lá cây quá nhỏ chứa nhiễu thị trường
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,    # Mỗi cây chỉ lấy ngẫu nhiên 50% số lượng cột để tăng tính đa dạng
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'seed': 42,
    'n_jobs': -1,               # Tận dụng toàn bộ các Core CPU của máy chủ Kaggle
    'verbose': -1
}
print("⚙️ Cấu hình Hyperparameters LightGBM đã sẵn sàng.")

⚙️ Cấu hình Hyperparameters LightGBM đã sẵn sàng.


In [6]:
import joblib

# Mảng lưu trữ kết quả dự đoán Out-of-Fold (OOF) của toàn bộ hệ thống
oof_predictions = np.zeros(len(df))

# Tạo thư mục lưu weights mô hình trên Kaggle
models_dir = './models/lgbm'
os.makedirs(models_dir, exist_ok=True)

# Danh sách 5 hạt giống chiến lược của giải pháp vô địch
SEEDS = [0, 11, 42, 777, 2021]

# Kích hoạt trình chia Fold chuỗi thời gian nghiêm ngặt
folds_generator = splitter.split(df)

for fold, (train_idx, val_idx) in enumerate(folds_generator):
    print(f"\n" + "="*40 + f" 🌲 HUẤN LUYỆN FOLD {fold + 1}/4 " + "="*40)
    
    # Phân tách tập dữ liệu Train/Validation theo đúng dòng thời gian thực lịch sử
    X_train, y_train = df.loc[train_idx, features], df.loc[train_idx, 'target']
    X_val, y_val = df.loc[val_idx, features], df.loc[val_idx, 'target']
    
    # 💡 THỦ THUẬT QUANTS: Tính toán trọng số phạt sai số.
    # Vì bài toán tính điểm bằng metric RMSPE, ta lấy nghịch đảo bình phương của nhãn mục tiêu làm weight.
    # Cây quyết định sẽ tập trung tối ưu các vùng có giá trị Volatility nhỏ - nơi dễ bị phạt nặng nhất.
    weight_train = 1 / np.square(y_train)
    weight_val = 1 / np.square(y_val)
    
    # Ma trận lưu trữ kết quả của 5 hạt giống trong fold hiện tại
    fold_preds_seeds = np.zeros((len(X_val), len(SEEDS)))
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed {seed}...")
        current_params = params.copy()
        current_params['seed'] = seed
        
        # Đóng gói dữ liệu vào cấu trúc Dataset đặc thù của LightGBM
        train_data = lgb.Dataset(X_train, label=y_train, weight=weight_train)
        val_data = lgb.Dataset(X_val, label=y_val, weight=weight_val, reference=train_data)
        
        # Kích hoạt hàm train native API để áp dụng Custom Metric feval_rmspe
        model = lgb.train(
            params=current_params,
            train_set=train_data,
            num_boost_round=1500,
            valid_sets=[train_data, val_data],
            feval=feval_rmspe, # Thước đo RMSPE custom từ Mốc 9
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False), # Tự động dừng nếu 50 vòng không cải thiện
                lgb.log_evaluation(period=0) # Tắt in log dòng dày đặc giúp Notebook sạch sẽ, tăng tốc độ chạy
            ]
        )
        
        # Ghi nhận dự đoán của hạt giống này trên tập Validation
        fold_preds_seeds[:, seed_idx] = model.predict(X_val)
        
        # Đóng gói và lưu Model Weights xuống phân vùng /models/lgbm/
        model_save_path = f"{models_dir}/lgbm_fold{fold+1}_seed{seed}.pkl"
        joblib.dump(model, model_save_path)
        
        del train_data, val_data, model
        gc.collect()
        
    # Lấy trung bình cộng kết quả từ cả 5 hạt giống để làm đại diện cuối cùng cho Fold này
    val_preds_avg = np.mean(fold_preds_seeds, axis=1)
    oof_predictions[val_idx] = val_preds_avg
    
    # Tính điểm kiểm định RMSPE Out-of-Fold của riêng Fold này
    fold_score = rmspe(y_val.values, val_preds_avg)
    print(f"🎯 Điểm RMSPE OOF hoàn thành cho Fold {fold + 1}: {fold_score:.5f}")
    
    del X_train, y_train, X_val, y_val, weight_train, weight_val, fold_preds_seeds
    gc.collect()


======================================== 🌲 HUẤN LUYỆN FOLD 1/4 ========================================
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 0...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 11...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 42...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 777...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 2021...
🎯 Điểm RMSPE OOF hoàn thành cho Fold 1: 0.23588

======================================== 🌲 HUẤN LUYỆN FOLD 2/4 ========================================
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 0...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 11...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 42...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 777...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 2021...
🎯 Điểm RMSPE OOF hoàn thành cho Fold 2: 0.22014

======================================== 🌲 HUẤN LUYỆN FOLD 3/4 ========================================
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 0...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 11...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 42...
   ↳ 🚀 Đang chạy cỗ máy GBDT - Seed 777.

In [7]:
# Chỉ tính toán trên những chỉ mục (indexes) thực sự nằm trong các khối validation Out-of-Fold
valid_indexes = oof_predictions > 0
final_oof_score = rmspe(df.loc[valid_indexes, 'target'].values, oof_predictions[valid_indexes])

print("\n" + "═"*60)
print(f"🏆 ĐIỂM SỐ KIỂM ĐỊNH TOÀN HỆ THỐNG (FINAL CV OOF RMSPE): {final_oof_score:.5f}")
print("═"*60)
print("Weight của 20 mô hình LightGBM (4 Folds x 5 Seeds) đã được lưu trữ an toàn trong `./models/lgbm/`")


════════════════════════════════════════════════════════════
🏆 ĐIỂM SỐ KIỂM ĐỊNH TOÀN HỆ THỐNG (FINAL CV OOF RMSPE): 0.22733
════════════════════════════════════════════════════════════
Weight của 20 mô hình LightGBM (4 Folds x 5 Seeds) đã được lưu trữ an toàn trong `./models/lgbm/`
